# Power BI Dashboard — Build & DAX Reference

This notebook contains step-by-step instructions and ready-to-copy DAX measures to build the four-page Power BI dashboard for your Churn Analysis project. Follow the 'How to see the dashboard' section to view it in Power BI Desktop.

## What I added

- Clear checklist of actions to build the report in Power BI Desktop
- Date table DAX (copy into `Modeling -> New Table`)
- Core KPI and trend DAX measures (copy into `Modeling -> New Measure`)
- Page-by-page visual recommendations and quick setup steps

## Prerequisites

1. Install and open Power BI Desktop (latest stable release).
2. Locate your cleaned dataset file (e.g., cleaned_churn_data.csv). Use the cleaned file — do not modify it.
3. Optional: `reports/feature_importance.csv` will be used for the Feature Importance visual on Page 4.
4. Recommended import mode: `Import` for datasets up to ~1M rows for best interactivity.

## Import dataset into Power BI

1. In Power BI Desktop: `Home -> Get data -> Text/CSV` and select your cleaned CSV file.
2. Inspect in `Power Query` (Transform Data) only for metadata types; do not alter rows/values unless correcting obvious type issues.
3. `Close & Apply` to load the table. The table name shown here is assumed to be `CustomerData` — rename if you imported under a different name.

## Create Date table (Modeling -> New Table)

Copy the DAX below and paste into `Modeling -> New Table` to create a Date dimension. Replace `CustomerData[SignupDate]` and `CustomerData[ChurnDate]` if your column names differ.

```DAX
Date = CALENDAR( MIN( CustomerData[SignupDate] ), MAX( COALESCE(CustomerData[ChurnDate], TODAY()) ) )
// Add recommended columns after creating the table: Year, Month, MonthNumber, YearMonth = FORMAT([Date], "yyyy-MM"), Quarter
```

## Core DAX measures (copy each into `Modeling -> New Measure`)

Use these measures as-is, replacing column/table names if necessary.

```DAX
Total Customers = DISTINCTCOUNT( CustomerData[CustomerID] )

Active Customers = CALCULATE( DISTINCTCOUNT( CustomerData[CustomerID] ), CustomerData[Churn] = "No" )

Churned Customers = CALCULATE( DISTINCTCOUNT( CustomerData[CustomerID] ), CustomerData[Churn] = "Yes" )

Churn Rate = DIVIDE( [Churned Customers], [Total Customers], 0 )

Total Revenue = SUM( CustomerData[Revenue] )

Revenue Lost = CALCULATE( SUM( CustomerData[Revenue] ), CustomerData[Churn] = "Yes" )

ARPU = DIVIDE( [Total Revenue], [Total Customers], 0 )

Avg Tenure (months) = AVERAGE( CustomerData[TenureMonths] )

// Monthly churn (by ChurnDate using USERELATIONSHIP) -- replace Date[Date] names if different
Monthly Churn Count = CALCULATE( DISTINCTCOUNT( CustomerData[CustomerID] ), CustomerData[Churn] = "Yes", USERELATIONSHIP( Date[Date], CustomerData[ChurnDate] ) )

Monthly Revenue = SUM( CustomerData[MonthlyRevenue] )
```

## Page-by-page quick setup (create these visuals in Report view)

**Page 1 — Executive Overview**: KPI cards for the 8 measures above; `Stacked Column` for Revenue by Plan; `Treemap` for Segment Distribution; `Line chart` for Monthly Revenue Trend; `Area/Line` for Churn Trend. Add slicers for Gender, Plan, ContractType, AgeGroup.

**Page 2 — Customer Analytics**: Churn by AgeGroup/Gender/Contract/Plan (Stacked/Clustered column charts); Tenure distribution (histogram).

**Page 3 — Revenue Analytics**: Revenue by Plan, Revenue Lost (filter Churn = Yes), ARPU by Plan (use `ARPU` measure), Top customers table (use Top N filter).

**Page 4 — Churn Insights**: Complaints vs Churn (stacked column); Churn by Tenure (line or binned column); Feature Importance (import `reports/feature_importance.csv` and show as bar chart); optional churn risk histogram if model scores available.

## How to see the dashboard (step-by-step)

1. Open Power BI Desktop.
2. `Home -> Get data -> Text/CSV` -> select your cleaned CSV file (for example `cleaned_churn_data.csv` or `data/exported_churn_data.csv`).
3. `Transform Data` only if you need to fix data types; otherwise `Close & Apply`.
4. Create the `Date` table (Modeling -> New Table) using the DAX above.
5. Create measures (Modeling -> New Measure) and paste each DAX measure from the 'Core DAX measures' cell.
6. Build Report pages: add visuals, set measure fields, configure Top N and filters, and sync slicers across pages (View -> Sync slicers).
7. Save the report as a `.pbix` file for portfolio submission and export a PDF snapshot via `File -> Export -> Export to PDF` if needed.
8. To include the PBIX in GitHub, add a short `README.md` describing how to open the PBIX and include screenshots; do not commit the raw CSV if it contains sensitive data — instead include a small sample or instructions to generate it.

## Feature importance (Page 4)

1. `Home -> Get data -> Text/CSV` -> select `reports/feature_importance.csv`.
2. Load it as `FeatureImportance`. Create a bar chart with `Feature` on axis and `Importance` as value sorted descending.
3. This table is display-only and does not need relationships to `CustomerData` unless you join feature values for advanced drillthroughs.